# Análisis por nodo del PML (CENACE) — solar vs punta, producción solar y precio capturado

Este notebook responde, **nodo por nodo** (y después agrupado por **región** o **estado**):

1. ¿Cuánto vale la energía en horario **solar** (06:00-18:00) y en horario **punta** (18:00-22:00)?
2. Si pongo **1 kWp solar** en un nodo, ¿cuánto produce **cada hora** y a qué precio se vende esa energía?
3. ¿Qué tan distinto es eso de multiplicar "irradiancia anual × precio promedio" (el método *Total Generation* que usa REPSA en prospección)?

**Cómo usarlo:** edita la celda de *Configuración* y corre todo con **Kernel → Restart & Run All**.
Tarda ~1-2 minutos (la parte más lenta es calcular la producción solar de todos los nodos; se puede apagar).

**Requisito extra (una sola vez):** en la terminal de VS Code, con tu `.venv` activado: `pip install pvlib`

*Se basa en `Reporte_Nodos.ipynb` (composición, spread, canibalización) y añade: tabla solar/punta por nodo, agrupación por región/estado, producción hora por hora y precio capturado. No incluye baterías.*

*A kWp stands for kilowatt-peak, which measures the maximum power output of a solar panel system under ideal laboratory conditions.*

## Paso 0 — Configuración

| Parámetro | Qué hace |
|---|---|
| `NODOS` | Nodos que quieres ver en detalle (código CENACE). |
| `AGRUPAR_POR` | `"region"` = Centro de Control Regional de CENACE (Noroeste, Norte, Noreste, Occidental, Central, Oriental, Peninsular) · `"estado"` = entidad federativa. |
| `DESDE`, `HASTA` | Periodo a analizar. Recomendado: 12 meses completos, para que todas las estaciones pesen igual. |
| `MONEDA` | `"MXN"` (como publica CENACE) o `"USD"`. |
| `TIPO_CAMBIO` | MXN por USD. Solo se usa si `MONEDA = "USD"`. |
| `NODO_EJEMPLO`, `FECHA_EJEMPLO` | Nodo y día para el ejemplo hora por hora del Paso 5. |
| `CALCULAR_TODOS_LOS_NODOS` | `True` = calcula la producción solar de los ~2,450 nodos (≈1 min) para poder agrupar por región/estado. |

In [ ]:
NODOS = ["03MFP-400", "04AGD-230", "05CHU-230", "08MDP-230"]   # SLP, Sonora, Chihuahua, Yucatán
AGRUPAR_POR = "region"            # "region" o "estado"
DESDE, HASTA = "2025-08-01", "2026-07-31"

MONEDA = "MXN"                    # "MXN" o "USD"
TIPO_CAMBIO = 17.84               # MXN por USD (Banxico FIX, 28-sep-2026)

HORAS_SOLAR = (6, 18)             # 06:00-18:00  -> hora >= 6  AND hora < 18
HORAS_PUNTA = (18, 22)            # 18:00-22:00  -> hora >= 18 AND hora < 22

NODO_EJEMPLO = "03MFP-400"        # Maniobras Fotovoltaico Potosí (San Luis Potosí)
FECHA_EJEMPLO = "2026-04-15"

CALCULAR_TODOS_LOS_NODOS = True
# FUENTE_PRODUCCION = "cielo_despejado"   # "cielo_despejado" (sin internet) o "nasa" (ver Paso 4)
FUENTE_PRODUCCION = "nasa"   # "cielo_despejado" (sin internet) o "nasa" (ver Paso 4)

In [ ]:
import os
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from conexion import conectar      # abre db/pml.duckdb y crea la vista precios_pml (hora corregida)

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

con = conectar(os.path.join("db", "pml.duckdb"))
os.makedirs("salidas", exist_ok=True)

COL_GRUPO = "region" if AGRUPAR_POR == "region" else "estado"
UNIDAD = f"{MONEDA}/MWh"
COLORWAY = ["#2563eb", "#dc2626", "#16a34a", "#d97706", "#7c3aed", "#0891b2", "#db2777", "#65a30d"]


def a_moneda(x):
    # CENACE publica en MXN/MWh; si se pide USD dividimos entre el tipo de cambio
    return x / TIPO_CAMBIO if MONEDA == "USD" else x


fmin, fmax = con.execute("SELECT MIN(fecha), MAX(fecha) FROM precios_pml").fetchone()
print(f"Datos disponibles: {fmin} a {fmax} · periodo elegido: {DESDE} a {HASTA} · precios en {UNIDAD}")

## Paso 1 — La corrección de horas (hora CENACE − 1)

CENACE numera las horas del **1 al 24** y cada número es la hora en que **termina** el intervalo:
la "hora 1" es de 00:00 a 01:00, la "hora 7" es de 06:00 a 07:00.

La tabla `pml_hourly` guarda los datos tal como vienen de CENACE. Para analizar usamos la vista
`precios_pml` (definida en `conexion.py`), donde **hora = hora CENACE − 1**, o sea la hora en que
**empieza** el intervalo (0 a 23). Así "horario solar 06:00-18:00" se escribe `hora >= 6 AND hora < 18`.

La celda de abajo muestra las primeras horas de un día en ambas versiones: el precio es el mismo, solo cambia la etiqueta.

In [ ]:
crudo = con.execute(
    "SELECT hora AS hora_cenace, pml FROM pml_hourly WHERE nodo = ? AND fecha = ? ORDER BY hora LIMIT 4",
    [NODO_EJEMPLO, FECHA_EJEMPLO]).df()
corregido = con.execute(
    "SELECT hora AS hora_corregida, pml FROM precios_pml WHERE nodo = ? AND fecha = ? ORDER BY hora LIMIT 4",
    [NODO_EJEMPLO, FECHA_EJEMPLO]).df()
tabla = pd.concat([crudo, corregido[["hora_corregida"]]], axis=1)
tabla["intervalo"] = [f"{h:02d}:00-{h+1:02d}:00" for h in tabla["hora_corregida"]]
tabla[["hora_cenace", "hora_corregida", "intervalo", "pml"]]

## Paso 2 — Catálogo: ¿en qué región y estado está cada nodo?

Viene de la tabla `nodos_ubicacion` (catálogo de NodosP de CENACE + centroides de municipio de INEGI,
generada con `geolocalizar_nodos.py`). La ubicación es la **cabecera municipal**, no la subestación exacta.

In [ ]:
ubic = con.execute(
    "SELECT nodo, nombre, centro_control_regional AS region, entidad_federativa AS estado, "
    "municipio, lat, lon FROM nodos_ubicacion").df().drop_duplicates("nodo").set_index("nodo")

faltan = [n for n in NODOS + [NODO_EJEMPLO] if n not in ubic.index]
assert not faltan, f"Estos nodos no están en el catálogo: {faltan}"
display(ubic.loc[NODOS, ["nombre", "region", "estado", "municipio"]])
print("Nodos por región (catálogo):")
ubic["region"].value_counts()


## Paso 3 — Solar vs punta, por nodo

Para cada nodo, en el periodo elegido:

- **PML solar** = promedio del PML de todas las horas de 06:00 a 18:00 de todos los días.
- **PML punta** = promedio del PML de 18:00 a 22:00.
- **Spread** = punta − solar (en dinero) y punta ÷ solar − 1 (en %).

Se calcula para **todos** los nodos en una sola consulta (así después se puede agrupar), y abajo se muestran los que elegiste.

In [ ]:
q = f'''
    SELECT nodo,
           COUNT(DISTINCT fecha)                                                   AS dias,
           AVG(pml)                                                                AS pml_24h,
           AVG(pml) FILTER (WHERE hora >= {HORAS_SOLAR[0]} AND hora < {HORAS_SOLAR[1]}) AS pml_solar,
           AVG(pml) FILTER (WHERE hora >= {HORAS_PUNTA[0]} AND hora < {HORAS_PUNTA[1]}) AS pml_punta
    FROM precios_pml
    WHERE fecha BETWEEN ? AND ?
    GROUP BY nodo
'''
sp = con.execute(q, [DESDE, HASTA]).df()
for c in ["pml_24h", "pml_solar", "pml_punta"]:
    sp[c] = a_moneda(sp[c])
sp["spread"] = sp["pml_punta"] - sp["pml_solar"]
sp["spread_pct"] = 100 * (sp["pml_punta"] / sp["pml_solar"].where(sp["pml_solar"] > 0) - 1)
sp = sp.join(ubic[["nombre", "region", "estado", "municipio"]], on="nodo")

sel = sp[sp["nodo"].isin(NODOS)].set_index("nodo").loc[NODOS]
display(sel[["nombre", "region", "estado", "dias", "pml_24h", "pml_solar", "pml_punta", "spread", "spread_pct"]])

larga = sel.reset_index().melt(id_vars="nodo", value_vars=["pml_solar", "pml_punta"],
                               var_name="bloque", value_name="precio")
fig = px.bar(larga, x="nodo", y="precio", color="bloque", barmode="group", text_auto=".0f",
             color_discrete_sequence=COLORWAY,
             labels={"precio": f"PML promedio ({UNIDAD})", "bloque": ""},
             title=f"PML solar (06-18h) vs punta (18-22h) · {DESDE} a {HASTA}")
fig.show()

### Solar vs punta, por región o estado

Promedio **simple entre nodos** del grupo (cada nodo pesa igual; no tenemos volúmenes de energía por nodo
para ponderar). `spread_p10` y `spread_p90` muestran la dispersión dentro del grupo: si están muy separados,
elegir bien el nodo dentro de la región importa mucho.

In [ ]:
grp = (sp.dropna(subset=[COL_GRUPO])
         .groupby(COL_GRUPO)
         .agg(nodos=("nodo", "count"), pml_24h=("pml_24h", "mean"), pml_solar=("pml_solar", "mean"),
              pml_punta=("pml_punta", "mean"), spread=("spread", "mean"),
              spread_p10=("spread", lambda s: s.quantile(0.10)), spread_p90=("spread", lambda s: s.quantile(0.90)),
              spread_pct=("spread_pct", "mean"))
         .sort_values("spread", ascending=False))
display(grp)

larga_g = grp.reset_index().melt(id_vars=COL_GRUPO, value_vars=["pml_solar", "pml_punta"],
                                 var_name="bloque", value_name="precio")
px.bar(larga_g, x=COL_GRUPO, y="precio", color="bloque", barmode="group", text_auto=".0f",
       color_discrete_sequence=COLORWAY, labels={"precio": f"PML promedio ({UNIDAD})", "bloque": ""},
       title=f"PML solar vs punta por {AGRUPAR_POR}").show()

px.box(sp.dropna(subset=[COL_GRUPO]), x=COL_GRUPO, y="spread", points="all", hover_data=["nodo", "nombre"],
       labels={"spread": f"Spread punta − solar ({UNIDAD})"},
       title=f"Spread de cada nodo, agrupado por {AGRUPAR_POR} (cada punto es un nodo)").show()

### Evolución mensual de solar, punta y spread (nodos elegidos)

In [ ]:
q = f'''
    SELECT nodo, strftime(make_date(anio, mes, 1), '%Y-%m') AS mes,
           AVG(pml) FILTER (WHERE hora >= {HORAS_SOLAR[0]} AND hora < {HORAS_SOLAR[1]}) AS pml_solar,
           AVG(pml) FILTER (WHERE hora >= {HORAS_PUNTA[0]} AND hora < {HORAS_PUNTA[1]}) AS pml_punta
    FROM precios_pml
    WHERE list_contains(?, nodo) AND fecha BETWEEN ? AND ?
    GROUP BY nodo, anio, mes ORDER BY nodo, mes
'''
ev = con.execute(q, [NODOS, DESDE, HASTA]).df()
for c in ["pml_solar", "pml_punta"]:
    ev[c] = a_moneda(ev[c])
ev["spread"] = ev["pml_punta"] - ev["pml_solar"]
px.line(ev, x="mes", y="spread", color="nodo", markers=True, color_discrete_sequence=COLORWAY,
        labels={"spread": f"Spread punta − solar ({UNIDAD})"}, title="Spread mensual por nodo").show()
px.line(ev, x="mes", y="pml_solar", color="nodo", markers=True, color_discrete_sequence=COLORWAY,
        labels={"pml_solar": f"PML solar 06-18h ({UNIDAD})"}, title="PML en horario solar, mensual").show()

## Paso 4 — ¿De dónde sale la producción solar hora por hora?

CENACE da **precios**, no producción. La producción de un panel en cada nodo la **modelamos** con la librería
[pvlib](https://pvlib-python.readthedocs.io) (Sandia National Laboratories), en tres pasos:

1. **Posición del sol**: con la latitud/longitud del nodo (cabecera municipal) y la fecha-hora, pvlib calcula
   dónde está el sol en el cielo en cada momento. Es geometría astronómica: no es un supuesto.
2. **Irradiancia**: cuánta radiación llega. Aquí hay dos opciones (`FUENTE_PRODUCCION`):
   - `"cielo_despejado"`: modelo de **día sin nubes** (Ineichen). Funciona sin internet. La *forma* del día
     (a qué hora sale el sol, cuándo es el pico, cuánto dura el día en invierno vs verano) es correcta, pero
     el *total* sale alto porque ignora nubes y lluvias (p. ej. el temporal de junio-septiembre).
   - `"nasa"`: irradiancia **medida por satélite, hora por hora**, de [NASA POWER](https://power.larc.nasa.gov).
     Es la misma fuente que REPSA ya usa en la presentación *Office prospection calculations*
     (ahí se descarga a mano en la web, promedio anual). Aquí se descargaría automáticamente por hora. Incluye
     nubes → más realista. Necesita internet (córrelo en tu Mac).
3. **Panel**: tracker de un eje norte-sur (típico en plantas grandes en México), que sigue al sol de este a oeste.

Resultado: para cada nodo, una tabla **días × 24 horas** con los **kWh que produce 1 kWp** en cada hora.

**Hora de referencia:** las horas de CENACE parecen estar todas en **hora del centro (UTC-6)**. En Sonora el
valle de precios del mediodía aparece ~1.5 h más tarde que en Occidente, lo que coincide con que ahí el sol
"llega tarde" respecto al reloj del centro. Por eso la producción se calcula en UTC-6 para todos los nodos.

**Datos faltantes de NASA (importante).** NASA publica la radiación con semanas de retraso: en octubre de
2026 todavía no tiene julio de 2026 (lo entrega como `-999`). Un día con alguna hora sin dato **se excluye
completo** del cálculo de producción y precio capturado (no se cuenta como día sin sol). Las columnas
`dias_con_produccion` y `cobertura_pct` dicen cuántos días se usaron. Cuando NASA publique esos meses, el
código los descarga solo (vuelve a intentar cada 3 días).

**Caché.** Lo descargado se guarda en `salidas/cache_nasa/celdas/`, un archivo por celda de 1° (NASA da la
misma radiación en todo un cuadro de ~100 x 100 km). Es la misma caché que usa `valor_nodo.py`; antes de
usarla se revisa qué fechas faltan y solo se descargan esas.

In [ ]:
try:
    import pvlib
except ImportError:
    raise ImportError("Falta pvlib. En la terminal (con tu .venv activo): pip install pvlib")

# Las piezas del modelo viven en valor_nodo.py, así el notebook y el script dan exactamente lo mismo:
#   _timestamps  -> momentos UTC del punto medio de cada hora CENACE (hora del centro, UTC-6)
#   _nasa_power  -> radiación y temperatura de NASA POWER, con caché por celda y NaN donde no hay dato
#   _poa_tracker -> posición del sol + radiación sobre un tracker de 1 eje (+ pérdida por temperatura)
from valor_nodo import _timestamps, _nasa_power, _poa_tracker

UTC_OFFSET = -6                                  # hora del centro
CACHE_NASA = os.path.join("salidas", "cache_nasa")
NASA_OFFLINE = False                  # True = no intentar descargar; usar solo lo guardado


def perfil_fv(lat, lon, fechas):
    """kWh producidos por 1 kWp en cada hora -> matriz (días, 24). Días sin dato de NASA = NaN."""
    momentos = _timestamps(fechas, UTC_OFFSET)
    lugar = pvlib.location.Location(lat, lon, tz="UTC")
    if FUENTE_PRODUCCION == "nasa":
        s = _nasa_power(lat, lon, fechas, CACHE_NASA, offline=NASA_OFFLINE).reindex(
            momentos, method="nearest", tolerance=pd.Timedelta(minutes=31))
        poa = _poa_tracker(lugar, momentos, ghi=s["ALLSKY_SFC_SW_DWN"], temp=s["T2M"])
    else:
        poa = _poa_tracker(lugar, momentos)       # cielo despejado
    E = poa.values.reshape(len(fechas), 24) / 1000.0   # W/m2 durante 1 h -> kWh por kWp
    E[np.isnan(E).any(axis=1)] = np.nan               # día con alguna hora sin dato -> día sin dato
    return E


print("pvlib", pvlib.__version__, "· fuente de producción:", FUENTE_PRODUCCION)

## Paso 5 — Ejemplo de UN día, hora por hora

Aquí está la respuesta a "¿cómo se hace la suma hora por hora?". Para el nodo y día de ejemplo:

| Columna | Qué es |
|---|---|
| `pml` | Precio de esa hora en el nodo (CENACE) |
| `kwh_por_kwp` | Energía que produce 1 kWp en esa hora (modelo del Paso 4) |
| `ingreso` | `pml × kwh_por_kwp ÷ 1000` → lo que se cobra por esa energía (el ÷1000 pasa de MWh a kWh) |

- **Promedio simple del día** = suma de los 24 PML ÷ 24 → cada hora pesa igual, aunque de noche no produzcas nada.
- **Precio capturado del día** = suma de `pml × kwh` ÷ suma de `kwh` → cada hora pesa **según cuánto produces**.
  Es un **promedio ponderado** del PML, *no* es el PML de una hora. Si el precio se hunde justo al mediodía
  (cuando más produces), el precio capturado queda **por debajo** del promedio simple.

In [ ]:
fechas_ej = pd.DatetimeIndex([pd.Timestamp(FECHA_EJEMPLO)])
u = ubic.loc[NODO_EJEMPLO]
precios_dia = con.execute(
    "SELECT hora, pml FROM precios_pml WHERE nodo = ? AND fecha = ? ORDER BY hora",
    [NODO_EJEMPLO, FECHA_EJEMPLO]).df()
assert len(precios_dia) == 24, "Ese día no tiene 24 horas de datos; elige otra FECHA_EJEMPLO"

ej = precios_dia.copy()
ej["intervalo"] = [f"{h:02d}:00-{h+1:02d}:00" for h in ej["hora"]]
ej["pml"] = a_moneda(ej["pml"])
ej["kwh_por_kwp"] = perfil_fv(u["lat"], u["lon"], fechas_ej)[0]
assert not ej["kwh_por_kwp"].isna().any(), "NASA no tiene radiación para ese día; elige otra FECHA_EJEMPLO"
ej["ingreso"] = ej["pml"] * ej["kwh_por_kwp"] / 1000          # por kWp instalado

display(ej[["hora", "intervalo", "pml", "kwh_por_kwp", "ingreso"]].round({"pml": 1, "kwh_por_kwp": 3, "ingreso": 3}))

prom_simple = ej["pml"].mean()
captura_dia = (ej["pml"] * ej["kwh_por_kwp"]).sum() / ej["kwh_por_kwp"].sum()
print(f"Nodo {NODO_EJEMPLO} ({u['nombre']}, {u['estado']}) · {FECHA_EJEMPLO}")
print(f"  Energía del día          = {ej['kwh_por_kwp'].sum():.2f} kWh por kWp")
print(f"  Ingreso del día          = {ej['ingreso'].sum():.3f} {MONEDA} por kWp")
print(f"  Promedio simple del PML  = {prom_simple:,.1f} {UNIDAD}   (24 horas, mismo peso)")
print(f"  Precio capturado del día = {ej['ingreso'].sum():.3f} / {ej['kwh_por_kwp'].sum():.2f} x 1000 = {captura_dia:,.1f} {UNIDAD}")
print(f"  Captura / promedio       = {captura_dia / prom_simple:.2f}")

In [ ]:
fig = go.Figure()
fig.add_bar(x=ej["hora"], y=ej["kwh_por_kwp"], name="Producción (kWh por kWp)", marker_color="#f59e0b", yaxis="y2")
fig.add_scatter(x=ej["hora"], y=ej["pml"], name=f"PML ({UNIDAD})", mode="lines+markers", line=dict(color="#2563eb"))
fig.add_hline(y=prom_simple, line_dash="dot", annotation_text="promedio simple")
fig.add_hline(y=captura_dia, line_dash="dash", line_color="#dc2626", annotation_text="precio capturado")
fig.update_layout(title=f"{NODO_EJEMPLO} · {FECHA_EJEMPLO}: precio y producción por hora",
                  xaxis=dict(title="Hora de inicio del intervalo", dtick=1),
                  yaxis=dict(title=f"PML ({UNIDAD})"),
                  yaxis2=dict(title="kWh por kWp", overlaying="y", side="right", showgrid=False))
fig.show()

## Paso 6 — Precio capturado y *value factor* del periodo completo, por nodo

Lo mismo del Paso 5, pero sumando **todas las horas de todos los días** del periodo:

- **Precio capturado** = Σ (PML × kWh) ÷ Σ kWh, sobre todas las horas del periodo.
- **PML promedio 24 h** = promedio simple de todas las horas del periodo.
- **Value factor (VF)** = precio capturado ÷ PML promedio 24 h **del periodo completo** (no de cada día).
  VF = 0.80 significa: la energía solar se vende, en promedio, 20 % más barata que el precio promedio del nodo.
- **% energía a PML ≤ 0** = qué parte de los kWh producidos se vendería a precio cero o negativo.
- **kWh por kWp (modelo)** = producción anual del modelo. Con `nasa` es realista (~1,900-2,350 en México); con `cielo_despejado` es un **techo sin nubes**: úsalo solo para comparar.
- **Días sin dato de NASA** se excluyen de *todo* lo de esta sección (producción **y** precio), para que el precio capturado y el PML promedio se comparen sobre los mismos días. `cobertura_pct` = % de los días del periodo que se usaron.

In [ ]:
HC = [f"h{h:02d}" for h in range(24)]


def matriz_precios(nodos=None):
    # Una fila por (nodo, día) con los 24 precios en columnas h00..h23; solo días completos
    cols = ", ".join(f"MAX(pml) FILTER (WHERE hora = {h}) AS h{h:02d}" for h in range(24))
    filtro, params = ("AND list_contains(?, nodo)", [DESDE, HASTA, list(nodos)]) if nodos else ("", [DESDE, HASTA])
    df = con.execute(f'''
        SELECT nodo, fecha, COUNT(*) AS n, {cols}
        FROM precios_pml WHERE fecha BETWEEN ? AND ? {filtro}
        GROUP BY nodo, fecha HAVING COUNT(*) = 24''', params).df()
    df["fecha"] = pd.to_datetime(df["fecha"])
    return df


def captura(mat):
    fechas = pd.date_range(DESDE, HASTA, freq="D")
    pos = {f: i for i, f in enumerate(fechas)}
    perfiles, filas = {}, []
    for nodo, g in mat.groupby("nodo"):
        if nodo not in ubic.index or pd.isna(ubic.at[nodo, "lat"]):
            continue
        lat, lon = round(ubic.at[nodo, "lat"], 3), round(ubic.at[nodo, "lon"], 3)
        if (lat, lon) not in perfiles:                       # nodos del mismo municipio comparten perfil
            perfiles[(lat, lon)] = perfil_fv(lat, lon, fechas)
        E = perfiles[(lat, lon)][g["fecha"].map(pos).values]   # kWh por kWp (días x 24)
        P = a_moneda(g[HC].values.astype(float))                 # precio   (días x 24)
        dias = len(g)
        ok = ~np.isnan(E).any(axis=1)          # días con producción conocida (NASA puede no tenerlos)
        E, P, n = E[ok], P[ok], int(ok.sum())
        fila = {"nodo": nodo, "dias": dias, "dias_con_produccion": n, "cobertura_pct": 100 * n / dias}
        if n > 0 and E.sum() > 0:
            fila.update({
                "pml_24h": P.mean(),                                  # mismos días que la producción
                "precio_capturado": (E * P).sum() / E.sum(),
                "pct_energia_pml_le0": 100 * (E * (P <= 0)).sum() / E.sum(),
                "kwh_por_kwp_modelo": E.sum() * 365 / n,              # anualizado con los días usados
                "ingreso_por_kwp_modelo": (E * P).sum() / 1000 * 365 / n,   # MONEDA por kWp al año
            })
        filas.append(fila)
    out = pd.DataFrame(filas)
    out["value_factor"] = out["precio_capturado"] / out["pml_24h"]
    if (out["cobertura_pct"] < 100).any():
        print(f"Aviso: cobertura media de producción {out['cobertura_pct'].mean():.1f} % "
              f"({(out['cobertura_pct'] < 100).sum()} de {len(out)} nodos con días excluidos por falta de dato de NASA)")
    return out.join(ubic[["nombre", "region", "estado"]], on="nodo")


cap_sel = captura(matriz_precios(NODOS)).set_index("nodo").loc[NODOS]
display(cap_sel[["nombre", "region", "cobertura_pct", "pml_24h", "precio_capturado", "value_factor",
                 "pct_energia_pml_le0", "kwh_por_kwp_modelo", "ingreso_por_kwp_modelo"]])

larga = cap_sel.reset_index().melt(id_vars="nodo", value_vars=["pml_24h", "precio_capturado"],
                                   var_name="medida", value_name="precio")
px.bar(larga, x="nodo", y="precio", color="medida", barmode="group", text_auto=".0f",
       color_discrete_sequence=COLORWAY, labels={"precio": UNIDAD, "medida": ""},
       title="PML promedio 24 h vs precio capturado por un proyecto solar").show()

### Precio capturado por región o estado

Requiere `CALCULAR_TODOS_LOS_NODOS = True` (calcula la producción de ~800 ubicaciones distintas, ≈1 min).
En la gráfica de dispersión cada punto es un nodo: los que quedan **debajo de la diagonal** capturan menos
que su promedio (VF < 1). Entre más abajo, más canibalización solar.

In [ ]:
if CALCULAR_TODOS_LOS_NODOS:
    cap_todos = captura(matriz_precios())
    cap_grp = (cap_todos.dropna(subset=[COL_GRUPO]).groupby(COL_GRUPO)
               .agg(nodos=("nodo", "count"), pml_24h=("pml_24h", "mean"),
                    precio_capturado=("precio_capturado", "mean"), value_factor=("value_factor", "mean"),
                    vf_min=("value_factor", "min"), vf_max=("value_factor", "max"),
                    pct_energia_pml_le0=("pct_energia_pml_le0", "mean"))
               .sort_values("value_factor"))
    display(cap_grp)

    px.bar(cap_grp.reset_index(), x=COL_GRUPO, y="value_factor", text_auto=".2f",
           title=f"Value factor promedio por {AGRUPAR_POR} (1.0 = el solar cobra el precio promedio)").show()

    lim = [0, cap_todos["pml_24h"].quantile(0.99) * 1.05]
    fig = px.scatter(cap_todos, x="pml_24h", y="precio_capturado", color=COL_GRUPO,
                     hover_data=["nodo", "nombre", "estado", "value_factor"],
                     labels={"pml_24h": f"PML promedio 24 h ({UNIDAD})", "precio_capturado": f"Precio capturado ({UNIDAD})"},
                     title="Cada punto es un nodo · debajo de la línea = el solar captura menos que el promedio")
    fig.add_scatter(x=lim, y=lim, mode="lines", line=dict(dash="dot", color="grey"), name="captura = promedio")
    fig.update_xaxes(range=lim); fig.update_yaxes(range=lim)
    fig.show()
else:
    print("CALCULAR_TODOS_LOS_NODOS = False → se omite la agrupación del precio capturado.")

## Paso 7 — Comparación con el método *Total Generation* de REPSA

En la presentación interna **"6. Origination – Office prospection calculations"** (carpeta OneDrive de REPSA),
la prioridad de un sitio se calcula así:

> **Total Generation** = irradiancia anual (kWh/m² al año, de NASA POWER) × precio del nodo (USD/MWh) ÷ 1000
>
> Ejemplo de la presentación: 40.73 USD/MWh × 2,220 kWh/m² ÷ 1000 = **90**

Ese número mezcla dos cosas **anuales**: cuánta energía hay y cuánto vale el MWh **en promedio**. Implícitamente
supone que el solar vende cada kWh al precio promedio del nodo, o sea que **VF = 1**.

La versión hora por hora (Paso 6) multiplica cada hora por su propio precio. La diferencia entre ambos es
exactamente el value factor:

  método anual ÷ método horario = 1 ÷ VF

Si VF = 0.60, el método anual **sobreestima el ingreso en 67 %** (1/0.60 − 1). La tabla usa la misma producción
en los dos métodos para que la única diferencia sea el precio.

In [ ]:
base = cap_todos if CALCULAR_TODOS_LOS_NODOS else cap_sel.reset_index()
comp = base[["nodo", "nombre", "region", "estado", "pml_24h", "kwh_por_kwp_modelo",
             "ingreso_por_kwp_modelo", "value_factor"]].copy()
comp["total_generation_repsa"] = comp["pml_24h"] * comp["kwh_por_kwp_modelo"] / 1000   # precio promedio × energía
comp["sobreestimacion_pct"] = 100 * (comp["total_generation_repsa"] / comp["ingreso_por_kwp_modelo"] - 1)

display(comp.set_index("nodo").loc[[n for n in NODOS if n in set(comp["nodo"])],
        ["nombre", "total_generation_repsa", "ingreso_por_kwp_modelo", "value_factor", "sobreestimacion_pct"]])
if CALCULAR_TODOS_LOS_NODOS:
    display(comp.groupby(COL_GRUPO)["sobreestimacion_pct"].mean().sort_values().to_frame("sobreestimación media %"))

## Paso 8 — Exportar tablas (para graficar en Excel, Streamlit o PowerPoint)

In [ ]:
sufijo = f"{DESDE}_{HASTA}_{MONEDA}"
sp.to_csv(f"salidas/solar_punta_nodos_{sufijo}.csv", index=False)
grp.to_csv(f"salidas/solar_punta_{AGRUPAR_POR}_{sufijo}.csv")
cap_sel.to_csv(f"salidas/captura_nodos_elegidos_{sufijo}.csv")
if CALCULAR_TODOS_LOS_NODOS:
    cap_todos.to_csv(f"salidas/captura_todos_nodos_{sufijo}.csv", index=False)
    cap_grp.to_csv(f"salidas/captura_{AGRUPAR_POR}_{sufijo}.csv")
print("Guardado en la carpeta salidas/:", sorted(f for f in os.listdir("salidas") if sufijo in f))

## Nota: ¿MXN o USD?

- **Usa MXN para analizar el mercado.** Es la moneda en que CENACE publica y liquida. Convertir agrega "ruido"
  del tipo de cambio: si el peso se mueve 5 %, tus precios en USD se mueven 5 % sin que el mercado haya cambiado.
- **Usa USD cuando compares contra costos o te lo pidan socios:** CAPEX de paneles y equipos, PPAs en dólares,
  o la hoja de prospección de REPSA (que está en USD/MWh).
- Si conviertes, **dilo siempre** y usa un solo tipo de cambio documentado (aquí `TIPO_CAMBIO`, Banxico FIX).
  Para series de varios años lo más riguroso es el tipo de cambio promedio de cada mes.